In [1]:
import os

print('Notebook folder:', os.getcwd())
print('Data folder exists:', os.path.exists('../data'))

Notebook folder: C:\Users\Admin\OneDrive\Desktop\ecommerce _sales _ project\notebooks
Data folder exists: True


In [3]:
import os

for name in files.values():
    path = os.path.join('..', 'data', name)
    print(os.path.exists(path), name)

print()
print('Files actually in the data folder:')
for f in sorted(os.listdir('../data')):
    print(repr(f))

True olist_orders_dataset.csv
True olist_order_items_dataset.csv
True olist_order_payments_dataset.csv
True olist_customers_dataset.csv
True olist_products_dataset.csv
True olist_order_reviews_dataset.csv
True product_category_name_translation.csv

Files actually in the data folder:
'.ipynb_checkpoints'
'olist_customers_dataset.csv'
'olist_geolocation_dataset.csv'
'olist_order_items_dataset.csv'
'olist_order_payments_dataset.csv'
'olist_order_reviews_dataset.csv'
'olist_orders_dataset.csv'
'olist_products_dataset.csv'
'olist_sellers_dataset.csv'
'product_category_name_translation.csv'


In [4]:
import os
import getpass
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Password is typed at runtime, so it is never saved in this file
password = getpass.getpass('Enter MySQL password: ')

url = URL.create(
    drivername='mysql+pymysql',
    username='root',
    password=password,
    host='127.0.0.1',
    port=3306,
    database='olist',
)
engine = create_engine(url)

files = {
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

date_cols = {
    'orders': ['order_purchase_timestamp', 'order_approved_at',
               'order_delivered_carrier_date', 'order_delivered_customer_date',
               'order_estimated_delivery_date'],
}

for table, file in files.items():
    path = os.path.join('..', 'data', file)
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        # Some files contain non-UTF-8 characters (e.g. Portuguese review text)
        df = pd.read_csv(path, encoding='latin-1')
    for c in date_cols.get(table, []):
        # Dates in these CSVs are in DD-MM-YYYY HH:MM format
        df[c] = pd.to_datetime(df[c], format='%d-%m-%Y %H:%M', errors='coerce')
    df.to_sql(table, engine, if_exists='replace', index=False, chunksize=10000)
    print(table, df.shape)

Enter MySQL password:  ········


orders (99441, 8)
order_items (112650, 7)
order_payments (103886, 5)
customers (99441, 5)
products (32951, 9)
order_reviews (99224, 7)
category_translation (71, 2)


In [5]:
orders = pd.read_csv('../data/olist_orders_dataset.csv')
orders[['order_purchase_timestamp','order_delivered_customer_date']].head(10)

,order_purchase_timestamp,order_delivered_customer_date
0,02-10-2017 10:56,10-10-2017 21:25
1,24-07-2018 20:41,07-08-2018 15:27
2,08-08-2018 08:38,17-08-2018 18:06
3,18-11-2017 19:28,02-12-2017 00:28
4,13-02-2018 21:18,16-02-2018 18:17
5,09-07-2017 21:57,26-07-2017 10:57
6,11-04-2017 12:22,NaN
7,16-05-2017 13:10,26-05-2017 12:55
8,23-01-2017 18:29,02-02-2017 14:08
9,29-07-2017 11:55,16-08-2017 17:14
